# Automata-01 — Automates finis : reconnaître, déterminiser, minimiser

**Navigation** : [Index de la série](README.md) | [Version Python](Automata-01-AutomatesFinis-Python.ipynb)

> **Public : Découverte.** Jumeau C# du notebook `Automata-01`. Il ne suppose que la
> bibliothèque de base .NET (BCL) : collections, LINQ. Aucune notion d'automate n'est
> requise : AFD, AFN et minimisation sont définis ici, puis **vérifiés exhaustivement**.

## Ce qu'on y apprend

1. **Reconnaître** un mot par un automate fini (déterministe ou non), avec epsilon-transitions.
2. **Déterminiser** par la construction des sous-ensembles, en mesurant que le langage ne change pas.
3. **Minimiser** par l'algorithme de **Hopcroft**, et vérifier l'unicité de l'automate minimal.

**Vérification indépendante** : chaque construction est confrontée à une **référence
externe** (une propriété du langage, ou `EndsWith`) sur **tous** les mots jusqu'à une
longueur bornée. C'est la même méthode que le jumeau Python ; les résultats doivent
coïncider.


In [1]:
// En-tete du jumeau C# : BCL uniquement (aucun package NuGet), culture fr-FR epinglee.
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Text;
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.GetCultureInfo("fr-FR");
CultureInfo.CurrentCulture = CultureInfo.GetCultureInfo("fr-FR");
Console.WriteLine($"Kernel C# actif : .NET {Environment.Version} ; automates finis en BCL seule.");


Kernel C# actif : .NET 10.0.15 ; automates finis en BCL seule.


## 1. Un automate fini déterministe (AFD)

Un AFD est une machine à **états finis** qui lit un mot lettre à lettre, sans autre mémoire
que son état courant : états $Q$, alphabet $\Sigma$, transition
$\delta : Q \times \Sigma \to Q$, état initial $q_0$, états finaux $F$. Le mot est
**accepté** si la lecture finit dans un état final.

Le plus simple qui fasse quelque chose : sur $\{0, 1\}$, accepter les mots à **nombre
pair de `1`**.


In [2]:
// AFD "nombre pair de 1" : deux etats (0=pair, 1=impair). delta[(etat, lettre)] = suivant.
var deltaPair = new Dictionary<(int, char), int> {
    [(0, '0')] = 0, [(0, '1')] = 1,
    [(1, '0')] = 1, [(1, '1')] = 0,
};

bool AfdAccepte(Dictionary<(int, char), int> delta, int initial, HashSet<int> finals, string mot) {
    int q = initial;
    foreach (char c in mot) {
        if (!delta.TryGetValue((q, c), out q)) return false;  // transition absente = rejet
    }
    return finals.Contains(q);
}

var finalsPair = new HashSet<int> { 0 };
foreach (string mot in new[] { "", "0", "1", "101", "0110", "111", "1010" }) {
    bool ok = AfdAccepte(deltaPair, 0, finalsPair, mot);
    Console.WriteLine($"'{mot}', {(ok ? "accepte" : "rejete")}");
}


'', accepte


'0', accepte


'1', rejete


'101', accepte


'0110', accepte


'111', rejete


'1010', accepte


**Lecture.** Le mot vide est accepté (zéro est pair), `'101'` l'est,
`'111'` non. Deux états suffisent : la seule chose à retenir est la **parité** du nombre
de `1`. La référence externe est `mot.Count(c => c == '1') % 2 == 0` ; vérifions l'accord
sur **tous** les mots jusqu'à la longueur 10.


In [3]:
// Tous les mots sur un alphabet de longueur 0..nMax (mot vide inclus).
IEnumerable<string> Mots(char[] alphabet, int nMax) {
    yield return "";
    for (int n = 1; n <= nMax; n++) {
        foreach (int[] idx in Produits(alphabet.Length, n))
            yield return new string(idx.Select(i => alphabet[i]).ToArray());
    }
}
IEnumerable<int[]> Produits(int k, int n) {
    int[] cur = new int[n];
    while (true) {
        yield return cur.ToArray();
        int p = n - 1;
        while (p >= 0 && cur[p] == k - 1) { cur[p] = 0; p--; }
        if (p < 0) break;
        cur[p]++;
    }
}

var tous = Mots(new[] { '0', '1' }, 10).ToList();
int divergences = tous.Count(w => AfdAccepte(deltaPair, 0, finalsPair, w) != (w.Count(c => c == '1') % 2 == 0));
Console.WriteLine($"{tous.Count} mots testes (longueur <= 10)");
Console.WriteLine($"Divergences AFD / reference 'parite' : {divergences}");


2047 mots testes (longueur <= 10)


Divergences AFD / reference 'parite' : 0


**Lecture.** Zéro divergence sur 2047 mots : l'AFD et la définition en
intension décrivent le même langage — exactement le résultat du jumeau Python.


## 2. Le non-déterminisme et les epsilon-transitions

Un **AFN** mène à **plusieurs** états par lettre ($\delta : Q \times \Sigma \to
\mathcal{P}(Q)$) et autorise les **epsilon-transitions** ($\varepsilon$, sans lire de
lettre). Il accepte s'il **existe** un chemin réussi. Exemple : les mots sur $\{a, b\}$
**finissant par `abb`**.


In [4]:
// AFN : delta compose ((etat, lettre) -> HashSet d'etats), eps (etat -> HashSet).
class AfN {
    public Dictionary<(int, char), HashSet<int>> Delta = new();
    public Dictionary<int, HashSet<int>> Eps = new();
    public HashSet<char> Alphabet = new();
    public int Initial;
    public HashSet<int> Finals = new();

    public HashSet<int> EClosure(IEnumerable<int> S) {
        var res = new HashSet<int>(S);
        var pile = new Stack<int>(res);
        while (pile.Count > 0) {
            int q = pile.Pop();
            if (Eps.TryGetValue(q, out var cibles))
                foreach (int r in cibles)
                    if (res.Add(r)) pile.Push(r);
        }
        return res;
    }

    public bool Accepte(string mot) {
        var courant = EClosure(new[] { Initial });
        foreach (char c in mot) {
            var suivant = new HashSet<int>();
            foreach (int q in courant)
                if (Delta.TryGetValue((q, c), out var t)) suivant.UnionWith(t);
            courant = EClosure(suivant);
            if (courant.Count == 0) return false;
        }
        return courant.Overlaps(Finals);
    }
}

// (a|b)* abb : etat 0 boucle sur a,b ; on devine le 'a' qui ouvre le abb final.
var afnAbb = new AfN {
    Alphabet = new HashSet<char> { 'a', 'b' },
    Initial = 0,
    Finals = new HashSet<int> { 3 },
};
afnAbb.Delta[(0, 'a')] = new HashSet<int> { 0, 1 };
afnAbb.Delta[(0, 'b')] = new HashSet<int> { 0 };
afnAbb.Delta[(1, 'b')] = new HashSet<int> { 2 };
afnAbb.Delta[(2, 'b')] = new HashSet<int> { 3 };

foreach (string mot in new[] { "abb", "aabb", "babb", "abab", "abba", "ab", "" })
    Console.WriteLine($"'{mot}', {(afnAbb.Accepte(mot) ? "accepte" : "rejete")}");


'abb', accepte


'aabb', accepte


'babb', accepte


'abab', rejete


'abba', rejete


'ab', rejete


'', rejete


**Lecture.** L'AFN accepte exactement les mots finis par `abb`, avec 4
états. La référence externe sera `mot.EndsWith("abb")`, réutilisée après déterminisation.


## 3. Déterminiser : la construction des sous-ensembles

Tout AFN a un AFD équivalent (Rabin–Scott) : un **état** de l'AFD représente
**l'ensemble** des états où l'AFN *pourrait* être. On part de l'epsilon-clôture de
l'initial. Un AFN à $n$ états peut demander jusqu'à $2^n$ états ; mesurons la taille réelle.

En C#, un état composé est un `HashSet<int>` ; on le compare par **contenu** (clé texte
triée) plutôt que par référence.


In [5]:
// Cle canonique d'un ensemble d'etats (triee) : sert a identifier un etat compose.
string Cle(IEnumerable<int> S) => string.Join(",", S.OrderBy(x => x));

// Determinisation : etats identifies par leur cle ; deltaCompose[(cle, lettre)] = cle_suivante.
(List<HashSet<int>> etats, Dictionary<(string, char), string> delta, string init, HashSet<string> finals)
Determiniser(AfN afn) {
    string init = Cle(afn.EClosure(new[] { afn.Initial }));
    var etats = new Dictionary<string, HashSet<int>> { [init] = afn.EClosure(new[] { afn.Initial }) };
    var delta = new Dictionary<(string, char), string>();
    var finals = new HashSet<string>();
    var aTrait = new Stack<string>();
    aTrait.Push(init);
    while (aTrait.Count > 0) {
        string sk = aTrait.Pop();
        var S = etats[sk];
        if (S.Overlaps(afn.Finals)) finals.Add(sk);
        foreach (char c in afn.Alphabet) {
            var suivant = new HashSet<int>();
            foreach (int q in S)
                if (afn.Delta.TryGetValue((q, c), out var t)) suivant.UnionWith(t);
            var T = afn.EClosure(suivant);
            string tk = Cle(T);
            delta[(sk, c)] = tk;
            if (!etats.ContainsKey(tk)) { etats[tk] = T; aTrait.Push(tk); }
        }
    }
    return (etats.Values.ToList(), delta, init, finals);
}

bool AfdComposeAccepte(Dictionary<(string, char), string> delta, string init, HashSet<string> finals, string mot) {
    string q = init;
    foreach (char c in mot) {
        if (!delta.TryGetValue((q, c), out q)) return false;
    }
    return finals.Contains(q);
}

var (etatsAbb, deltaAbb, initAbb, finalsAbb) = Determiniser(afnAbb);
Console.WriteLine($"AFN : 4 etats -> AFD : {etatsAbb.Count} etats ; finaux : {finalsAbb.Count}");

var tousAb = Mots(new[] { 'a', 'b' }, 9).ToList();
int divAfn = tousAb.Count(w => afnAbb.Accepte(w) != AfdComposeAccepte(deltaAbb, initAbb, finalsAbb, w));
int divRef = tousAb.Count(w => AfdComposeAccepte(deltaAbb, initAbb, finalsAbb, w) != w.EndsWith("abb"));
Console.WriteLine($"{tousAb.Count} mots testes ; divergences AFD/AFN : {divAfn} ; AFD/reference : {divRef}");


AFN : 4 etats -> AFD : 4 etats ; finaux : 1


1023 mots testes ; divergences AFD/AFN : 0 ; AFD/reference : 0


**Lecture.** 4 états déterministes (comme en Python), **zéro divergence**
sur 1023 mots, contre l'AFN et contre `EndsWith("abb")`. La déterminisation préserve le
langage. Cet AFD est-il minimal ?


## 4. Minimiser : l'algorithme de Hopcroft

L'AFD minimal d'un langage régulier est **unique à isomorphisme près** (Myhill–Nerode).
Hopcroft part de la partition $\{F, Q \setminus F\}$ et la raffine : deux états restent
ensemble tant qu'aucune lettre ne les sépare. Pour le voir **réduire**, partons d'un AFD
redondant : les mots sur $\{a\}$ de **longueur paire**, écrits avec **6** états.


In [6]:
// Hopcroft sur un AFD a etats entiers 0..n-1 ; retourne les classes (listes d'etats).
List<List<int>> Hopcroft(int n, Dictionary<(int, char), int> delta, HashSet<int> finals, char[] alphabet) {
    var tous = Enumerable.Range(0, n).ToHashSet();
    var P = new List<HashSet<int>>();
    var F = new HashSet<int>(finals);
    var nonF = new HashSet<int>(tous); nonF.ExceptWith(F);
    if (F.Count > 0) P.Add(F);
    if (nonF.Count > 0) P.Add(nonF);
    var W = P.Select(b => new HashSet<int>(b)).ToList();
    while (W.Count > 0) {
        var A = W[0]; W.RemoveAt(0);
        foreach (char c in alphabet) {
            var X = new HashSet<int>(tous.Where(q => delta.TryGetValue((q, c), out int t) && A.Contains(t)));
            var nouvelle = new List<HashSet<int>>();
            foreach (var Y in P) {
                var inter = new HashSet<int>(Y); inter.IntersectWith(X);
                var diff = new HashSet<int>(Y); diff.ExceptWith(X);
                if (inter.Count > 0 && diff.Count > 0) {
                    nouvelle.Add(inter); nouvelle.Add(diff);
                    var yW = W.FirstOrDefault(w => w.SetEquals(Y));
                    if (yW != null) { W.Remove(yW); W.Add(new HashSet<int>(inter)); W.Add(new HashSet<int>(diff)); }
                    else W.Add(inter.Count <= diff.Count ? new HashSet<int>(inter) : new HashSet<int>(diff));
                } else nouvelle.Add(Y);
            }
            P = nouvelle;
        }
    }
    return P.Select(c => c.OrderBy(x => x).ToList()).ToList();
}

// AFD "longueur paire" redondant : 6 etats en cycle, finaux = indices pairs.
var delta6 = new Dictionary<(int, char), int>();
for (int q = 0; q < 6; q++) delta6[(q, 'a')] = (q + 1) % 6;
var finals6 = new HashSet<int> { 0, 2, 4 };
var classes = Hopcroft(6, delta6, finals6, new[] { 'a' });
Console.WriteLine($"AFD initial : 6 etats ; classes de Hopcroft : {classes.Count}");
foreach (var c in classes.OrderBy(c => c[0]))
    Console.WriteLine($"  classe [{string.Join(", ", c)}]");


AFD initial : 6 etats ; classes de Hopcroft : 2


  classe [0, 2, 4]


  classe [1, 3, 5]


**Lecture.** 6 états → **2 classes** (paires / impaires), comme en
Python : Hopcroft découvre la périodicité 2 cachée dans la 6. Vérifions l'**unicité** :
deux AFD du langage « nombre pair de `1` » doivent donner le même nombre de classes.


In [7]:
// Langage "parite du nombre de 1", presente en 4 etats redondants : minimal = 2.
var delta4 = new Dictionary<(int, char), int> {
    [(0,'0')]=0,[(0,'1')]=1,[(1,'0')]=1,[(1,'1')]=2,
    [(2,'0')]=2,[(2,'1')]=3,[(3,'0')]=3,[(3,'1')]=0,
};
var finals4 = new HashSet<int> { 0, 2 };
var classesPair = Hopcroft(4, delta4, finals4, new[] { '0', '1' });
Console.WriteLine($"parite (version 4 etats) : {classesPair.Count} classes ; attendu 2");
Console.WriteLine(classesPair.Count == 2
    ? "Unicite verifiee : le nombre de classes est un invariant du langage."
    : "VIOLATION de l'unicite");


parite (version 4 etats) : 2 classes ; attendu 2


Unicite verifiee : le nombre de classes est un invariant du langage.


**Lecture.** 2 classes, comme la version à 2 états : le nombre de
classes est un invariant du **langage**, pas de l'automate. Même conclusion qu'en Python.

## Ce que le dépôt fait déjà avec les automates

- **`Search-10-SymbolicAutomata`** : automates symboliques (transitions = prédicats).
- **`Lean-14-Finiteness-Derivatives`** (+ companion Lean `14b`) : dérivées de Brzozowski,
  route algébrique vers la déterminisation, avec preuve formelle.
- **`SL-10-ActiveAutomataLearning`** : L\* apprend un AFD par requêtes.
- **`Z3-04-Strings-Regex`** : le solveur Z3 raisonne sur les regex — pont vers
  `Automata-02` (théorème de Kleene).

## Bilan

- Un AFD ne retient que son **état courant** ; un AFN ajoute non-déterminisme et epsilon.
- **Déterminiser** préserve le langage (0 divergence sur 1023 mots, AFN **et** référence).
- **Minimiser** (Hopcroft) donne l'AFD **unique** du langage (invariance du nombre de classes).

Les jumeaux Python et C# produisent des résultats **identiques** : 2047 et 1023 mots
vérifiés sans divergence, 6→2 classes sur le cas redondant, minimal unique confirmé.


## Exercices

Trois exercices, dans l'ordre du notebook. Chaque cellule s'exécute **sans erreur telle
qu'elle est livrée** : remplacez les lignes `// TODO etudiant` par votre solution.


### Exercice 1 — Écrire un AFD pour un langage donné

Construisez un AFD sur $\{0, 1\}$ qui accepte les mots se **terminant par `1`**.
Deux états suffisent. Remplissez `deltaEx1` et `finalsEx1` (état initial `0`).


In [8]:
// TODO etudiant : completer les transitions et les etats finaux.
var deltaEx1 = new Dictionary<(int, char), int> {
    // [(0, '0')] = ..., [(0, '1')] = ...,
    // [(1, '0')] = ..., [(1, '1')] = ...,
};
var finalsEx1 = new HashSet<int>();   // TODO etudiant

// Verification (reference : EndsWith("1")). A executer apres votre construction.
if (deltaEx1.Count > 0 && finalsEx1.Count > 0) {
    var tous = Mots(new[] { '0', '1' }, 9).ToList();
    var mauvais = tous.Where(w => AfdAccepte(deltaEx1, 0, finalsEx1, w) != w.EndsWith("1")).ToList();
    Console.WriteLine($"{tous.Count} mots testes ; divergences : {mauvais.Count}");
} else {
    Console.WriteLine("Exercice a completer : remplissez deltaEx1 et finalsEx1.");
}


Exercice a completer : remplissez deltaEx1 et finalsEx1.


### Exercice 2 — Déterminiser un AFN avec epsilon-transitions

Cet AFN reconnaît les mots sur $\{a, b\}$ **finissant par** `aa` **ou par** `b`.
Déterminisez-le avec `Determiniser`, puis vérifiez contre
`mot.EndsWith("aa") || mot.EndsWith("b")`. Combien d'états l'AFD a-t-il ?


In [9]:
var afnEx2 = new AfN {
    Alphabet = new HashSet<char> { 'a', 'b' },
    Initial = 0,
    Finals = new HashSet<int> { 2, 3 },
};
afnEx2.Delta[(0, 'a')] = new HashSet<int> { 0, 1 };
afnEx2.Delta[(0, 'b')] = new HashSet<int> { 0, 3 };
afnEx2.Delta[(1, 'a')] = new HashSet<int> { 2 };
afnEx2.Eps[2] = new HashSet<int> { 0 };   // apres 'aa', retour gratuit a 0

// TODO etudiant : utiliser Determiniser(afnEx2) puis AfdComposeAccepte pour verifier.
// var (etatsE2, deltaE2, initE2, finalsE2) = Determiniser(afnEx2);
// Console.WriteLine($"AFD : {etatsE2.Count} etats");
// var tous = Mots(new[] { 'a', 'b' }, 8).ToList();
// int mauvais = tous.Count(w => AfdComposeAccepte(deltaE2, initE2, finalsE2, w)
//     != (w.EndsWith("aa") || w.EndsWith("b")));
// Console.WriteLine($"Divergences : {mauvais}");
Console.WriteLine("Exercice a completer : decommentez le bloc ci-dessus.");

Exercice a completer : decommentez le bloc ci-dessus.


### Exercice 3 — Pourquoi $\{a^n b^n\}$ n'est pas régulier

Exercice de raisonnement (pas de code). **Expliquez en une phrase** pourquoi deux
préfixes $a^i$ et $a^j$ ($i \ne j$) sont **discernables**, et ce que cela implique pour
le nombre d'états d'un hypothétique AFD de $L = \{a^n b^n\}$. *(Indice : quel suffixe
distingue $a^i$ de $a^j$ ?)*


In [10]:
// Reponse a rediger en commentaire (exercice de raisonnement).
// TODO etudiant : a^i et a^j (i != j) sont discernables par le suffixe ...
// ... donc le nombre de classes de Myhill-Nerode est ..., donc un AFD de L aurait ...
Console.WriteLine("Exercice de raisonnement : redigez votre reponse en commentaire ci-dessus.");


Exercice de raisonnement : redigez votre reponse en commentaire ci-dessus.
